In [27]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np

df_convenios = pd.read_excel(r'C:\Users\alejandro_arboledas\Downloads\Convenios Actualizados (1) (1).xlsx')
df_convenios.shape

(73, 25)

In [28]:
df_convenios.columns.str.upper()

Index(['NOMBRE', 'N°', 'AÑO ', 'ESTADO', 'TIPO DE ALCANCE', 'PAÍS', 'CIUDAD',
       'CÓDIGO', 'UNIVERSIDAD/ ENTIDAD  ', 'TIPO DE CONVENIO', 'OBJETO ',
       'ACTIVIDADES/ACCIONES ESPECÍFICAS QUE CONTEMPLA EL CONVENIO',
       'POSEE DESCUENTOS, BENEFICIOS, BECAS U OTRO, CUÁL? ', 'SECTOR',
       'PROGRAMAS DE PREGRADO HOMÓLOGOS', 'PROGRAMAS DE POSGRADOS HOMÓLOGOS',
       'PERSONA DE CONTACTO', 'CORREO INSTITUCIONAL', 'ESTADO ', 'PRÓRROGA',
       'FECHA DE FIRMA', 'FECHA  DE TERMINACIÓN/RENOVACIÓN', 'ENLACE CONVENIO',
       'LINK PÁGINA', 'ENVIO CORREO INTERES COOPERACIÓN ACADÉMICA'],
      dtype='object')

In [29]:
df_convenios = df_convenios.apply(lambda col: col.str.upper() if col.dtype == 'object' else col)
df_convenios

,NOMBRE,N°,AÑO,ESTADO,TIPO DE ALCANCE,PAÍS,CIUDAD,CÓDIGO,UNIVERSIDAD/ ENTIDAD,TIPO DE CONVENIO,...,PROGRAMAS DE POSGRADOS HOMÓLOGOS,PERSONA DE CONTACTO,CORREO INSTITUCIONAL,ESTADO,PRÓRROGA,FECHA DE FIRMA,FECHA DE TERMINACIÓN/RENOVACIÓN,ENLACE CONVENIO,LINK PÁGINA,ENVIO CORREO INTERES COOPERACIÓN ACADÉMICA
0,ABEL NONATO ROYO GALEANO,12,2016,FIRMADO,INTERNACIONAL,CUBA,VILLA CLARA,CI 684,UNIVERSIDAD DE CIENCIAS MEDICAS VILLA CLARA,MARCO,...,TODOS LOS PROGRAMAS,EN CONSTRUCCIÓN,JULIOAGR@INFOMED.SLD.CU\n\nRELINTUCMVC@INFOMED...,ACTIVO,AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1WPD2RQEPHWHLY...,HTTPS://INSTITUCIONES.SLD.CU/UCMVC/,NO
1,ABEL NONATO ROYO GALEANO,14,2016,FIRMADO,INTERNACIONAL,PERÚ,TRUJILLO,CI 052,UNIVERSIDAD PRIVADA ANTENOR ORREGO,MARCO,...,TODOS LOS PROGRAMAS,MARTÍN ERNESTO POLO CUEVA,MPOLOC1@UPAO.EDU.PE,ACTIVO,AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1WC_RTDAUXFNNJ...,HTTPS://UPAO.EDU.PE/,SI
2,AREVALO GUILLIN WALTER,65,2024,FIRMADO,INTERNACIONAL,MÉXICO,GUADALAJARA,CI 2024 - 14 EXT,LOTTUS -UNIVERSIDAD TECNOLÓGICA DE GUADALAJARA,MARCO,...,TODOS LOS PROGRAMAS,GABRIELA BECERRA GOMEZ,ANDREA.GAMBOA@LOTTUSEDUCATION.COM,ACTIVO,DEFINIDO,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1WMHQB5ZUOAILF...,WWW.LOTTUS.COM,NaN
3,AREVALO GUILLIN WALTER,66,2024,FIRMADO,INTERNACIONAL,MÉXICO,LATACUNGA,CI MI 2024 - 02 EXT,UNIVERSIDAD TÉCNICA DE COTOPAXI,MARCO,...,TODOS LOS PROGRAMAS,DIEGO WLADIMIR CHANGO LLUMITASIG \n,DIEGO.CHANGO1415@UTC.EDU.EC/SUBDIRECCION.EXTEN...,ACTIVO,AUTOMÁTICA,28/03/2024,NaN,HTTPS://DRIVE.GOOGLE.COM/DRIVE/FOLDERS/13E5ZBO...,HTTPS://WWW.UTC.EDU.EC/,SI
4,AREVALO GUILLIN WALTER,76,2025,FIRMADO,INTERNACIONAL,ESPAÑA,ALMERÍA,CI 2025 - 12 EXT,UNIVERSIDAD DE ALMERÍA,MARCO,...,TODOS LOS PROGRAMAS,MANUEL CASTRO PÉREZ\nINTERNATIONAL RELATIONS O...,CRT@UAL.ES,ACTIVO,SIN PRÓRROGA AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/DRIVE/FOLDERS/1UZWWYA...,HTTPS://WWW.UAL.ES/,SI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,SANDRA MILENA MORENO GARCÍA,8,2015,FIRMADO,INTERNACIONAL,ESPAÑA,BARCELONA,CI 2023 - 02 EXT.,ASOCIACIÓN CULTURAL EL PARLANTE \n,MARCO,...,TODOS LOS PROGRAMAS,ALFREDO COHEN,ALFREDO@ELPARLANTE.ES,ACTIVO,AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/15UHWAETSVCCHR...,HTTPS://ELPARLANTE.ES/ES/,SI
69,SANDRA MILENA MORENO GARCÍA,9,2015,FIRMADO,INTERNACIONAL,ESPAÑA,MADRID,CI 645,UNIVERSIDAD A DISTANCIA DE MADRID,MARCO,...,TODOS LOS PROGRAMAS,FREDY WILBER ARAUJO,FREDYWILBER.ARAUJO@UDIMA.ES,ACTIVO,AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1JTWUYOXSQ_HBU...,HTTPS://UDIMA.CO/,SI
70,SANDRA MILENA MORENO GARCÍA,81,2025,FIRMADO,INTERNACIONAL,ESPAÑA,BARCELONA,CP 2025 03 EXT,UNIVERSIDAD AMERICANA DE EUROPA (UNADE),MARCO,...,"TRANSFORMACIÓN DIGITAL, GERENCIA DE LA MARCA, ...",ANDREA GÓMEZ MARTÍNEZ\nDIRECTORA DE INTERNACIO...,ANDREA.GOMEZ@UNADE.EDU.MX,ACTIVO,INDEFINIDO,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1CR6DA8SUXQ8IZ...,HTTPS://UNADE.EDU.MX/,NaN
71,WILLIAM LEONARDO OCHOA MORENO,26,2021,FIRMADO,INTERNACIONAL,ESPAÑA,MADRID,CI MI 2021 10 EXT,ESCUELA EUROPEA DE EDUCACIÓN Y EMPRESA- EUDE,MARCO,...,TODOS LOS PROGRAMAS,FEDERICO OSPINA,INFO@EUDE.ES / FEDERICO.OSPINA@EUDE.ES,ACTIVO,AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1EL2HKNCFTTP66...,HTTPS://LANDINGS.EUDE.CO,SI


In [30]:
df_convenios.columns = (
    df_convenios.columns
    .str.strip()
    .str.upper()
    .str.replace(r'\s*/\s*', '/', regex=True)
    .str.replace(r'\s*,\s*', ',', regex=True)
    .str.replace(' ', '_', regex=False)
)
df_convenios

,NOMBRE,N°,AÑO,ESTADO,TIPO_DE_ALCANCE,PAÍS,CIUDAD,CÓDIGO,UNIVERSIDAD/ENTIDAD,TIPO_DE_CONVENIO,...,PROGRAMAS_DE_POSGRADOS_HOMÓLOGOS,PERSONA_DE_CONTACTO,CORREO_INSTITUCIONAL,ESTADO,PRÓRROGA,FECHA_DE_FIRMA,FECHA__DE_TERMINACIÓN/RENOVACIÓN,ENLACE_CONVENIO,LINK_PÁGINA,ENVIO_CORREO_INTERES_COOPERACIÓN_ACADÉMICA
0,ABEL NONATO ROYO GALEANO,12,2016,FIRMADO,INTERNACIONAL,CUBA,VILLA CLARA,CI 684,UNIVERSIDAD DE CIENCIAS MEDICAS VILLA CLARA,MARCO,...,TODOS LOS PROGRAMAS,EN CONSTRUCCIÓN,JULIOAGR@INFOMED.SLD.CU\n\nRELINTUCMVC@INFOMED...,ACTIVO,AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1WPD2RQEPHWHLY...,HTTPS://INSTITUCIONES.SLD.CU/UCMVC/,NO
1,ABEL NONATO ROYO GALEANO,14,2016,FIRMADO,INTERNACIONAL,PERÚ,TRUJILLO,CI 052,UNIVERSIDAD PRIVADA ANTENOR ORREGO,MARCO,...,TODOS LOS PROGRAMAS,MARTÍN ERNESTO POLO CUEVA,MPOLOC1@UPAO.EDU.PE,ACTIVO,AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1WC_RTDAUXFNNJ...,HTTPS://UPAO.EDU.PE/,SI
2,AREVALO GUILLIN WALTER,65,2024,FIRMADO,INTERNACIONAL,MÉXICO,GUADALAJARA,CI 2024 - 14 EXT,LOTTUS -UNIVERSIDAD TECNOLÓGICA DE GUADALAJARA,MARCO,...,TODOS LOS PROGRAMAS,GABRIELA BECERRA GOMEZ,ANDREA.GAMBOA@LOTTUSEDUCATION.COM,ACTIVO,DEFINIDO,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1WMHQB5ZUOAILF...,WWW.LOTTUS.COM,NaN
3,AREVALO GUILLIN WALTER,66,2024,FIRMADO,INTERNACIONAL,MÉXICO,LATACUNGA,CI MI 2024 - 02 EXT,UNIVERSIDAD TÉCNICA DE COTOPAXI,MARCO,...,TODOS LOS PROGRAMAS,DIEGO WLADIMIR CHANGO LLUMITASIG \n,DIEGO.CHANGO1415@UTC.EDU.EC/SUBDIRECCION.EXTEN...,ACTIVO,AUTOMÁTICA,28/03/2024,NaN,HTTPS://DRIVE.GOOGLE.COM/DRIVE/FOLDERS/13E5ZBO...,HTTPS://WWW.UTC.EDU.EC/,SI
4,AREVALO GUILLIN WALTER,76,2025,FIRMADO,INTERNACIONAL,ESPAÑA,ALMERÍA,CI 2025 - 12 EXT,UNIVERSIDAD DE ALMERÍA,MARCO,...,TODOS LOS PROGRAMAS,MANUEL CASTRO PÉREZ\nINTERNATIONAL RELATIONS O...,CRT@UAL.ES,ACTIVO,SIN PRÓRROGA AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/DRIVE/FOLDERS/1UZWWYA...,HTTPS://WWW.UAL.ES/,SI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,SANDRA MILENA MORENO GARCÍA,8,2015,FIRMADO,INTERNACIONAL,ESPAÑA,BARCELONA,CI 2023 - 02 EXT.,ASOCIACIÓN CULTURAL EL PARLANTE \n,MARCO,...,TODOS LOS PROGRAMAS,ALFREDO COHEN,ALFREDO@ELPARLANTE.ES,ACTIVO,AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/15UHWAETSVCCHR...,HTTPS://ELPARLANTE.ES/ES/,SI
69,SANDRA MILENA MORENO GARCÍA,9,2015,FIRMADO,INTERNACIONAL,ESPAÑA,MADRID,CI 645,UNIVERSIDAD A DISTANCIA DE MADRID,MARCO,...,TODOS LOS PROGRAMAS,FREDY WILBER ARAUJO,FREDYWILBER.ARAUJO@UDIMA.ES,ACTIVO,AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1JTWUYOXSQ_HBU...,HTTPS://UDIMA.CO/,SI
70,SANDRA MILENA MORENO GARCÍA,81,2025,FIRMADO,INTERNACIONAL,ESPAÑA,BARCELONA,CP 2025 03 EXT,UNIVERSIDAD AMERICANA DE EUROPA (UNADE),MARCO,...,"TRANSFORMACIÓN DIGITAL, GERENCIA DE LA MARCA, ...",ANDREA GÓMEZ MARTÍNEZ\nDIRECTORA DE INTERNACIO...,ANDREA.GOMEZ@UNADE.EDU.MX,ACTIVO,INDEFINIDO,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1CR6DA8SUXQ8IZ...,HTTPS://UNADE.EDU.MX/,NaN
71,WILLIAM LEONARDO OCHOA MORENO,26,2021,FIRMADO,INTERNACIONAL,ESPAÑA,MADRID,CI MI 2021 10 EXT,ESCUELA EUROPEA DE EDUCACIÓN Y EMPRESA- EUDE,MARCO,...,TODOS LOS PROGRAMAS,FEDERICO OSPINA,INFO@EUDE.ES / FEDERICO.OSPINA@EUDE.ES,ACTIVO,AUTOMÁTICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1EL2HKNCFTTP66...,HTTPS://LANDINGS.EUDE.CO,SI


In [31]:
df = df_convenios.copy()

In [32]:
import unicodedata

def quitar_tildes(texto):
    if isinstance(texto, str):
        return ''.join(
            c for c in unicodedata.normalize('NFD', texto)
            if unicodedata.category(c) != 'Mn'
        )
    return texto

df.columns = (
    df.columns
    .str.strip()
    .str.upper()
    .str.replace(r'\s*/\s*', '/', regex=True)
    .str.replace(r'\s*,\s*', ',', regex=True)
    .str.replace(' ', '_', regex=False)
)

df.columns = [quitar_tildes(col) for col in df.columns]
df = df.apply(
    lambda col: col.map(quitar_tildes)
    if col.dtype == 'object'
    else col
)
df

,NOMBRE,N°,ANO,ESTADO,TIPO_DE_ALCANCE,PAIS,CIUDAD,CODIGO,UNIVERSIDAD/ENTIDAD,TIPO_DE_CONVENIO,...,PROGRAMAS_DE_POSGRADOS_HOMOLOGOS,PERSONA_DE_CONTACTO,CORREO_INSTITUCIONAL,ESTADO,PRORROGA,FECHA_DE_FIRMA,FECHA__DE_TERMINACION/RENOVACION,ENLACE_CONVENIO,LINK_PAGINA,ENVIO_CORREO_INTERES_COOPERACION_ACADEMICA
0,ABEL NONATO ROYO GALEANO,12,2016,FIRMADO,INTERNACIONAL,CUBA,VILLA CLARA,CI 684,UNIVERSIDAD DE CIENCIAS MEDICAS VILLA CLARA,MARCO,...,TODOS LOS PROGRAMAS,EN CONSTRUCCION,JULIOAGR@INFOMED.SLD.CU\n\nRELINTUCMVC@INFOMED...,ACTIVO,AUTOMATICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1WPD2RQEPHWHLY...,HTTPS://INSTITUCIONES.SLD.CU/UCMVC/,NO
1,ABEL NONATO ROYO GALEANO,14,2016,FIRMADO,INTERNACIONAL,PERU,TRUJILLO,CI 052,UNIVERSIDAD PRIVADA ANTENOR ORREGO,MARCO,...,TODOS LOS PROGRAMAS,MARTIN ERNESTO POLO CUEVA,MPOLOC1@UPAO.EDU.PE,ACTIVO,AUTOMATICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1WC_RTDAUXFNNJ...,HTTPS://UPAO.EDU.PE/,SI
2,AREVALO GUILLIN WALTER,65,2024,FIRMADO,INTERNACIONAL,MEXICO,GUADALAJARA,CI 2024 - 14 EXT,LOTTUS -UNIVERSIDAD TECNOLOGICA DE GUADALAJARA,MARCO,...,TODOS LOS PROGRAMAS,GABRIELA BECERRA GOMEZ,ANDREA.GAMBOA@LOTTUSEDUCATION.COM,ACTIVO,DEFINIDO,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1WMHQB5ZUOAILF...,WWW.LOTTUS.COM,NaN
3,AREVALO GUILLIN WALTER,66,2024,FIRMADO,INTERNACIONAL,MEXICO,LATACUNGA,CI MI 2024 - 02 EXT,UNIVERSIDAD TECNICA DE COTOPAXI,MARCO,...,TODOS LOS PROGRAMAS,DIEGO WLADIMIR CHANGO LLUMITASIG \n,DIEGO.CHANGO1415@UTC.EDU.EC/SUBDIRECCION.EXTEN...,ACTIVO,AUTOMATICA,28/03/2024,NaN,HTTPS://DRIVE.GOOGLE.COM/DRIVE/FOLDERS/13E5ZBO...,HTTPS://WWW.UTC.EDU.EC/,SI
4,AREVALO GUILLIN WALTER,76,2025,FIRMADO,INTERNACIONAL,ESPANA,ALMERIA,CI 2025 - 12 EXT,UNIVERSIDAD DE ALMERIA,MARCO,...,TODOS LOS PROGRAMAS,MANUEL CASTRO PEREZ\nINTERNATIONAL RELATIONS O...,CRT@UAL.ES,ACTIVO,SIN PRORROGA AUTOMATICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/DRIVE/FOLDERS/1UZWWYA...,HTTPS://WWW.UAL.ES/,SI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,SANDRA MILENA MORENO GARCIA,8,2015,FIRMADO,INTERNACIONAL,ESPANA,BARCELONA,CI 2023 - 02 EXT.,ASOCIACION CULTURAL EL PARLANTE \n,MARCO,...,TODOS LOS PROGRAMAS,ALFREDO COHEN,ALFREDO@ELPARLANTE.ES,ACTIVO,AUTOMATICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/15UHWAETSVCCHR...,HTTPS://ELPARLANTE.ES/ES/,SI
69,SANDRA MILENA MORENO GARCIA,9,2015,FIRMADO,INTERNACIONAL,ESPANA,MADRID,CI 645,UNIVERSIDAD A DISTANCIA DE MADRID,MARCO,...,TODOS LOS PROGRAMAS,FREDY WILBER ARAUJO,FREDYWILBER.ARAUJO@UDIMA.ES,ACTIVO,AUTOMATICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1JTWUYOXSQ_HBU...,HTTPS://UDIMA.CO/,SI
70,SANDRA MILENA MORENO GARCIA,81,2025,FIRMADO,INTERNACIONAL,ESPANA,BARCELONA,CP 2025 03 EXT,UNIVERSIDAD AMERICANA DE EUROPA (UNADE),MARCO,...,"TRANSFORMACION DIGITAL, GERENCIA DE LA MARCA, ...",ANDREA GOMEZ MARTINEZ\nDIRECTORA DE INTERNACIO...,ANDREA.GOMEZ@UNADE.EDU.MX,ACTIVO,INDEFINIDO,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1CR6DA8SUXQ8IZ...,HTTPS://UNADE.EDU.MX/,NaN
71,WILLIAM LEONARDO OCHOA MORENO,26,2021,FIRMADO,INTERNACIONAL,ESPANA,MADRID,CI MI 2021 10 EXT,ESCUELA EUROPEA DE EDUCACION Y EMPRESA- EUDE,MARCO,...,TODOS LOS PROGRAMAS,FEDERICO OSPINA,INFO@EUDE.ES / FEDERICO.OSPINA@EUDE.ES,ACTIVO,AUTOMATICA,NaN,NaN,HTTPS://DRIVE.GOOGLE.COM/FILE/D/1EL2HKNCFTTP66...,HTTPS://LANDINGS.EUDE.CO,SI


In [34]:
# SE RENOMBRAN COLUMNAS CON EL MISMO NOMBRE
new_cols = []
contador = 0

for col in df.columns:
    col_str = str(col)

    if col_str.startswith('ESTADO'):
        contador += 1
        if contador == 1:
            new_cols.append('ESTADO_1')
        elif contador == 2:
            new_cols.append('ESTADO_2')
        else:
            # 👇 MUY IMPORTANTE (para evitar el error)
            new_cols.append(f'ESTADO_{contador}')
    else:
        new_cols.append(col_str)

df.columns = new_cols

In [35]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'convenios'
schema_name = 'Internacionalizacion'
# Send the data to sql server
df.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 1000
)

# End connection
engine.dispose()